In [ ]:
# ============================================
# LV Index Project: Full Implementation (Steps 1–6)
# ============================================
# Uses:
#   - ACS_2011_2021.csv
#   - CRM Project(Sheet1).csv
#
# Output files (optional, at the bottom):
#   - lv_store_zips_with_features.csv
#   - lv_modeling_dataset.csv
#   - lv_variable_weights.csv
#   - lv_index_scores_by_zip.csv
# ============================================

import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# --------------------------------------------
# 0. Load input data
# --------------------------------------------
acs_path = "ACS_2011_2021.csv"
crm_path = "CRM Project(Sheet1).csv"

acs = pd.read_csv(acs_path)
# CRM has some non-UTF8 characters, so we use latin1
crm = pd.read_csv(crm_path, encoding="latin1")

print("ACS shape:", acs.shape)
print("CRM shape:", crm.shape)

# Ensure consistent ZIP code format: 5-character strings
acs["zip"] = acs["GEOID"].astype(str).str.zfill(5)

crm["zip"] = (
    crm["ZIP"]
    .astype(str)
    .str.extract(r"(\d{5})")[0]   # extract first 5-digit sequence, if any
)

crm = crm.dropna(subset=["zip"]).copy()
crm["zip"] = crm["zip"].str.zfill(5)

lv_zips = sorted(crm["zip"].unique())
print("Number of LV store ZIPs found in CRM:", len(lv_zips))

# --------------------------------------------
# 1. Prepare the LV Store ZIP Codes (112 ZIPs)
#    (Positive class in the model)
# --------------------------------------------
# We assume the 112 LV store ZIP codes are exactly the unique ZIPs in CRM.

# Filter ACS to latest year to avoid mixing across time
latest_year = acs["year"].max()
acs_latest = acs[acs["year"] == latest_year].copy()
acs_latest["zip"] = acs_latest["zip"].astype(str).str.zfill(5)

print(f"Using ACS data for year: {latest_year}")
print("ACS (latest year) shape:", acs_latest.shape)

# Flag which ACS rows correspond to LV store ZIPs
acs_latest["is_lv_zip"] = acs_latest["zip"].isin(lv_zips).astype(int)

lv_acs = acs_latest[acs_latest["is_lv_zip"] == 1].copy()
print("LV store ZIPs found in ACS (latest year):", lv_acs["zip"].nunique())

# --------------------------------------------
# 2. Build the Comparison ZIP Code Dataset
#    (City-level peers if possible; otherwise all non-LV ZIPs)
# --------------------------------------------

# All non-LV ZIPs in ACS are potential peers
peer_acs = acs_latest[acs_latest["is_lv_zip"] == 0].copy()
print("Total non-LV (peer) ZIPs:", peer_acs["zip"].nunique())

# ---- OPTIONAL: City-level peer filtering (requires additional ZIP↔city data)
# If you have a ZIP-to-city lookup for the whole US (e.g., zip_city_df with columns ["zip", "city"]),
# you could:
#
#   1) Merge acs_latest with zip_city_df to get a city for each ZIP.
#   2) Keep only peer ZIPs that share a city with at least one LV ZIP.
#
# For example:
#
#   acs_with_city = acs_latest.merge(zip_city_df, on="zip", how="left")
#   lv_cities = acs_with_city.loc[acs_with_city["is_lv_zip"] == 1, "city"].unique()
#   peer_acs = acs_with_city[
#       (acs_with_city["is_lv_zip"] == 0) &
#       (acs_with_city["city"].isin(lv_cities))
#   ].copy()
#
# In this script, we proceed without that extra lookup and treat all non-LV ZIPs as peers.

# --------------------------------------------
# 3. Select and Normalize Variables
# --------------------------------------------
# Choose a set of modeling variables that reflect:
#   - population size
#   - age structure
#   - income
#   - education
#   - poverty
#   - employment
#
# These column names are taken directly from the ACS file.

model_vars = [
    "population",
    "median_age",
    "median_household_income",
    "edu_bachelor_percent",
    "poverty_rate",
    "employment_rate_in_pop",
]

missing_vars = [v for v in model_vars if v not in acs_latest.columns]
if missing_vars:
    raise ValueError(f"The following model_vars are not in ACS: {missing_vars}")

# Build dataset containing both LV ZIPs and peer ZIPs for modeling
model_data = acs_latest[["zip", "is_lv_zip"] + model_vars].copy()
model_data = model_data.rename(columns={"is_lv_zip": "label"})

# Drop rows with any missing data in modeling variables
model_data = model_data.dropna(subset=model_vars + ["label"])
model_data["label"] = model_data["label"].astype(int)

print("Modeling dataset shape (after dropping NA):", model_data.shape)
print("LV ZIPs in modeling dataset:", model_data[model_data["label"] == 1]["zip"].nunique())
print("Peer ZIPs in modeling dataset:", model_data[model_data["label"] == 0]["zip"].nunique())

# Normalize variables to have mean 0, std 1 before logistic regression
X = model_data[model_vars].astype(float).values
y = model_data["label"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Store scaled features back into a copy for downstream calculations
model_data_scaled = model_data.copy()
for i, var in enumerate(model_vars):
    model_data_scaled[var] = X_scaled[:, i]

# --------------------------------------------
# 4. Run Logistic Regression
#    Dependent variable: label (1 = LV ZIP, 0 = non-LV)
#    Independent variables: normalized demographic variables
# --------------------------------------------
lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",  # help with class imbalance
    random_state=42
)
lr.fit(X_scaled, y)

print("Logistic regression fitted.")
print("Intercept:", lr.intercept_)
print("Coefficients (by variable):")
for var, coef in zip(model_vars, lr.coef_[0]):
    print(f"  {var}: {coef:.4f}")

# --------------------------------------------
# 5. Extract Coefficients as Weights
# --------------------------------------------
# We treat the absolute value of the coefficient as the magnitude of importance
# of that variable for distinguishing LV ZIPs from peers.

coef = lr.coef_[0]

weights = (
    pd.DataFrame({
        "variable": model_vars,
        "coefficient": coef,
    })
    .assign(abs_coefficient=lambda d: d["coefficient"].abs())
    .sort_values("abs_coefficient", ascending=False)
    .reset_index(drop=True)
)

print("\nTop variables by absolute coefficient (importance):")
print(weights)

# --------------------------------------------
# 6. Compute the Weighted Difference for Each Non-LV ZIP Code
# --------------------------------------------
# For each non-LV ZIP code:
#   - Compute the average (scaled) demographic values of all LV ZIPs
#   - For that ZIP, compute absolute difference from LV benchmark for each variable
#   - Multiply each difference by |coefficient| for that variable
#   - Sum these weighted differences to get a "distance" from LV-like profile
#   - Flip and scale to [0, 100] so higher LV index = more LV-like
# --------------------------------------------

# Boolean masks for LV vs peer rows
lv_mask = model_data_scaled["label"] == 1
peer_mask = model_data_scaled["label"] == 0

# 1) LV benchmark in scaled feature space
benchmark_scaled = model_data_scaled.loc[lv_mask, model_vars].mean(axis=0).values

# 2) Peer ZIP features in scaled space
peer_scaled = model_data_scaled.loc[peer_mask, model_vars].values
peer_zips = model_data_scaled.loc[peer_mask, "zip"].values

# 3) Absolute differences from LV benchmark for each variable
diff = np.abs(peer_scaled - benchmark_scaled)  # shape: (n_peers, n_features)

# 4) Use absolute coefficient values as weights
coef_weights = np.abs(coef)  # shape: (n_features,)

# Broadcast multiplication, then sum across variables
weighted_diff = diff * coef_weights  # shape: (n_peers, n_features)
lv_distance = weighted_diff.sum(axis=1)  # shape: (n_peers,)

# 5) Build a DataFrame of peer ZIP scores
peer_scores = pd.DataFrame({
    "zip": peer_zips,
    "lv_weighted_difference": lv_distance,
})

# 6) Convert "distance" to an LV Index where higher = more LV-like
d_min = peer_scores["lv_weighted_difference"].min()
d_max = peer_scores["lv_weighted_difference"].max()

if d_max == d_min:
    # Edge case: all distances identical
    peer_scores["lv_index"] = 100.0
else:
    # Min-max scale and flip
    peer_scores["lv_index"] = 100 * (
        1 - (peer_scores["lv_weighted_difference"] - d_min) / (d_max - d_min)
    )

# Sort so that the most LV-like ZIPs are at the top
peer_scores = (
    peer_scores
    .sort_values("lv_index", ascending=False)
    .reset_index(drop=True)
)

print("\nTop 20 non-LV ZIPs by LV Index:")
print(peer_scores.head(20))

# --------------------------------------------
# OPTIONAL: Save outputs to CSV
# --------------------------------------------
lv_acs_out = lv_acs[["zip"] + [c for c in model_vars if c in lv_acs.columns]].copy()
lv_acs_out.to_csv("lv_store_zips_with_features.csv", index=False)

model_data.to_csv("lv_modeling_dataset.csv", index=False)
weights.to_csv("lv_variable_weights.csv", index=False)
peer_scores.to_csv("lv_index_scores_by_zip.csv", index=False)

print("\nSaved:")
print("  lv_store_zips_with_features.csv")
print("  lv_modeling_dataset.csv")
print("  lv_variable_weights.csv")
print("  lv_index_scores_by_zip.csv")


ACS shape: (364998, 116)
CRM shape: (111, 17)
Number of LV store ZIPs found in CRM: 74
Using ACS data for year: 2021
ACS (latest year) shape: (33774, 117)
LV store ZIPs found in ACS (latest year): 73
Total non-LV (peer) ZIPs: 33701
Modeling dataset shape (after dropping NA): (30661, 8)
LV ZIPs in modeling dataset: 72
Peer ZIPs in modeling dataset: 30589
Logistic regression fitted.
Intercept: [-2.74660383]
Coefficients (by variable):
  population: 0.9638
  median_age: 0.3833
  median_household_income: -0.5835
  edu_bachelor_percent: 2.2649
  poverty_rate: 0.9897
  employment_rate_in_pop: 0.6369

Top variables by absolute coefficient (importance):
                  variable  coefficient  abs_coefficient
0     edu_bachelor_percent     2.264938         2.264938
1             poverty_rate     0.989721         0.989721
2               population     0.963766         0.963766
3   employment_rate_in_pop     0.636916         0.636916
4  median_household_income    -0.583463         0.583463
5   

In [ ]:
from google.colab import drive
drive.mount('/content/drive')